# Restaurant Turnover — V7 trial_1

**Student:** submission · **Register number:** submission · **Team:** submission

Predictor: **residual_tree_l2**, new-component weight **75%**.
Local validation status: **local champion**.

This is a model-generated experimental CSV, not a confirmed portal improvement.
The user-confirmed v5 score is **12,046,722.3691139**. Only a returned portal RMSE
can determine whether this candidate improves it. Select a CPU runtime in Colab,
run all cells, upload the four official CSVs when asked, and save the executed notebook.
Submit the final coding assignment only once, after choosing the best portal-scored CSV.

## Runtime and data

The notebook embeds the exact predictor implementation and frozen v5 reference.
It requires no earlier improvement folders. Checkpoints are saved in the output
folder; a browser disconnect can resume them while that runtime survives. For
persistence across runtime deletion, choose `USE_GOOGLE_DRIVE=True` below and
authorize Drive yourself. Free Colab runtime duration and resources are variable:
https://research.google.com/colaboratory/faq.html

In [ ]:
import os, sys, json, hashlib, base64, zlib, types, subprocess, importlib.metadata
from pathlib import Path

PACKAGES = {'numpy': '2.5.4', 'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'catboost': '1.2.10', 'scipy': '1.18.1'}
install = []
for name, version in PACKAGES.items():
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        install.append(f'{name}=={version}')
if install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *install])
import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib'])
    import matplotlib.pyplot as plt
print('Python:', sys.version.split()[0])
print({p: importlib.metadata.version(p) for p in PACKAGES})

In [ ]:
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['MLA_OUTPUT_DIR'] = '/content/drive/MyDrive/MLA_v7/trial_1'
DATA_DIR = Path(os.environ.get('MLA_DATA_DIR', '.')).resolve()
OUTPUT_DIR = Path(os.environ.get('MLA_OUTPUT_DIR', '.')).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED = ['Train_dataset.csv', 'Test_dataset.csv', 'Data_Dictionary.csv', 'Sample_Solution.csv']
if not all((DATA_DIR / n).is_file() for n in REQUIRED):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place the four official CSVs here, or set MLA_DATA_DIR.')
    files.upload()
    DATA_DIR = Path.cwd()
INPUT_HASHES = {n: hashlib.sha256((DATA_DIR / n).read_bytes()).hexdigest() for n in REQUIRED}

EXPECTED_INPUT_HASHES = {'Train_dataset.csv': 'ec40ebda74ba0bfbfe1af4417a4266a70eadf6e595500e99e5435e7bdc7ee51f', 'Test_dataset.csv': 'ee4137566adb1c0ce356a8c370e7b8a22f681bde492068efe11346d8d44b4f4d', 'Data_Dictionary.csv': '2d86c00e27448593564a161197b6639ee1eb90c01fa523b67e7a09d2ef405c5e', 'Sample_Solution.csv': '9519ba4d51eb2d59b8a71b054a62e1f4d48b72cf92930c31f5e5393a2cd135fb'}
assert INPUT_HASHES == EXPECTED_INPUT_HASHES, 'Input files differ from the measured experiment.'
print('PASS: four official input files and their SHA256 hashes.')

In [ ]:
REFERENCE_SOURCE = '"""Frozen v5 predictor, copied verbatim from the preserved implementations."""\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostRegressor\nfrom sklearn.model_selection import KFold, GroupKFold\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer\nfrom sklearn.linear_model import Ridge\nfrom sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor\nfrom sklearn.svm import SVR\n\n\nTARGET = \'Annual Turnover\'\n\nID = \'Registration Number\'\n\nEVENTS = [\'Live Music Rating\',\'Comedy Gigs Rating\',\'Value Deals Rating\',\'Live Sports Rating\']\n\nSOCIAL = [\'Facebook Popularity Quotient\',\'Instagram Popularity Quotient\']\n\nCORE = SOCIAL + [\'Hygiene Rating\',\'Restaurant Zomato Rating\',\'Live Music Rating\',\'Value for Money\',\'Overall Restaurant Rating\',\'Restaurant City Tier\',\'Cuisine\',\'Restaurant Theme\',\'Restaurant Location\',\'Endoresed By\',\'City\',\'Opening Year\']\n\ndef features(raw, variant=\'base\'):\n    data=raw.drop(columns=[TARGET,ID],errors=\'ignore\').copy()\n    dates=pd.to_datetime(data.pop(\'Opening Day of Restaurant\'),format=\'%d-%m-%Y\',errors=\'coerce\')\n    data[\'Opening Year\']=dates.dt.year.astype(float)\n    data[\'Opening Month\']=dates.dt.month.astype(float)\n    for c in EVENTS:\n        data[\'Offers \'+c.replace(\' Rating\',\'\')]=data[c].notna().astype(int)\n    parts=data[\'Cuisine\'].astype(\'string\').fillna(\'Missing\').str.split(\',\',n=1,expand=True)\n    data[\'Cuisine 1\']=parts[0].str.strip().str.lower()\n    data[\'Cuisine 2\']=parts.reindex(columns=[0,1])[1].astype(\'string\').str.strip().str.lower()\n    for c in [\'Cuisine\',\'City\',\'Restaurant Location\',\'Endoresed By\',\'Restaurant Type\',\'Restaurant Theme\']:\n        data[c]=data[c].astype(\'string\').str.strip().str.lower().str.replace(r\'\\s+\',\' \',regex=True).fillna(\'Missing\')\n    data[\'City Is Unknown\']=(data[\'City\']==\'-1\').astype(int)\n    for c in data.select_dtypes(include=[\'object\',\'string\']).columns:\n        data[c]=data[c].fillna(\'Missing\').astype(str)\n    if variant in [\'engineered\',\'engineered_drop\']:\n        data[\'City\']=data[\'City\'].replace({\'bengaluru\':\'bangalore\',\'gurugram\':\'gurgaon\',\'bombay\':\'mumbai\',\'calcutta\':\'kolkata\',\'delhi\':\'new delhi\'})\n        data[\'Cuisine unordered\']=[\'|\'.join(sorted(v.split(\',\'))) for v in data[\'Cuisine\']]\n        data[\'Opening Day\']=dates.dt.day.astype(float)\n        data[\'Opening Days Since 1970\']=(dates-pd.Timestamp(\'1970-01-01\')).dt.days.astype(float)\n        data[\'Social mean\']=data[SOCIAL].mean(axis=1)\n        data[\'Social difference\']=data[SOCIAL[0]]-data[SOCIAL[1]]\n        data[\'Social missing count\']=data[SOCIAL].isna().sum(axis=1)\n        data[\'Rating missing count\']=data.isna().sum(axis=1)\n        data[\'Event count\']=data[EVENTS].notna().sum(axis=1)\n        data[\'Event mean\']=data[EVENTS].mean(axis=1)\n        data[\'Quality mean\']=data[[\'Food Rating\',\'Hygiene Rating\',\'Overall Restaurant Rating\']].mean(axis=1)\n        data[\'Social x Zomato\']=data[\'Social mean\']*data[\'Restaurant Zomato Rating\']\n        data[\'Social x Hygiene\']=data[\'Social mean\']*data[\'Hygiene Rating\']\n        data[\'Quality x Value\']=data[\'Quality mean\']*data[\'Value for Money\']\n    if variant in [\'drop_shift\',\'engineered_drop\']:\n        data=data.drop(columns=[\'Service\',\'Comfortablility\'])\n    if variant==\'core\': data=data[CORE]\n    if variant==\'ratings\':\n        keep=SOCIAL+[\'Resturant Tier\',\'Restaurant Zomato Rating\',\'Restaurant City Tier\',\'Staff Responsivness\',\'Value for Money\',\'Hygiene Rating\',\'Overall Restaurant Rating\']+EVENTS+[\'Lively\',\'Ambience\',\'Privacy\']\n        data=data[keep]\n    if variant==\'no_city_date\':\n        data=data.drop(columns=[\'City\',\'City Is Unknown\',\'Opening Year\',\'Opening Month\'])\n    return data\n\ndef rmse(y,p): return float(np.sqrt(np.mean(np.square(np.asarray(y)-np.asarray(p)))))\n\ndef sklearn_model(X,spec,seed):\n    cats=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist()\n    nums=[c for c in X if c not in cats]\n    num=make_pipeline(SimpleImputer(strategy=\'median\',add_indicator=True),StandardScaler())\n    if spec[\'kind\']==\'spline\':\n        num=make_pipeline(SimpleImputer(strategy=\'median\',add_indicator=True),SplineTransformer(n_knots=4,degree=2,include_bias=False),StandardScaler())\n    pre=ColumnTransformer([(\'num\',num,nums),(\'cat\',OneHotEncoder(handle_unknown=\'ignore\',min_frequency=10,sparse_output=False),cats)])\n    kind=spec[\'kind\']\n    if kind in [\'ridge\',\'spline\']: model=Ridge(alpha=spec.get(\'alpha\',100))\n    elif kind==\'svr\': model=SVR(C=spec.get(\'C\',20),epsilon=.5,kernel=\'rbf\')\n    elif kind==\'hist\': model=HistGradientBoostingRegressor(max_iter=300,max_leaf_nodes=10,max_depth=4,learning_rate=.035,l2_regularization=30,min_samples_leaf=60,random_state=seed)\n    elif kind==\'extra\': model=ExtraTreesRegressor(n_estimators=400,min_samples_leaf=5,max_features=.7,n_jobs=4,random_state=seed)\n    return make_pipeline(pre,model)\n\ndef fit_predict(X,y,train_idx,valid_idx,spec,seed,fixed_iterations=None,return_fitted=False):\n    ys=np.asarray(y)[train_idx].copy()\n    if \'cap\' in spec: ys=np.minimum(ys,spec[\'cap\'])\n    if spec[\'kind\']==\'cat\':\n        params=dict(loss_function=spec.get(\'loss\',\'RMSE\'),eval_metric=\'RMSE\',iterations=fixed_iterations or spec.get(\'iterations\',1600),learning_rate=.04,depth=spec.get(\'depth\',6),l2_leaf_reg=spec.get(\'l2\',5),random_seed=seed,thread_count=4,verbose=False,allow_writing_files=False)\n        if \'boosting_type\' in spec: params[\'boosting_type\']=spec[\'boosting_type\']\n        if spec.get(\'monotone\'): params[\'monotone_constraints\']={**{c:1 for c in SOCIAL+[\'Hygiene Rating\',\'Restaurant Zomato Rating\']},\'Resturant Tier\':-1}\n        model=CatBoostRegressor(**params)\n        cat=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist()\n        stopping={} if fixed_iterations or \'iterations\' in spec else dict(eval_set=(X.iloc[valid_idx],np.asarray(y)[valid_idx]),early_stopping_rounds=120)\n        model.fit(X.iloc[train_idx],ys,cat_features=cat,**stopping)\n        p=model.predict(X.iloc[valid_idx]); trees=model.tree_count_\n    elif spec[\'kind\'] in [\'lgb\',\'xgb\']:\n        pre=sklearn_model(X,dict(kind=\'ridge\'),seed)[:-1]\n        ztr=pre.fit_transform(X.iloc[train_idx]); zva=pre.transform(X.iloc[valid_idx])\n        early=not (fixed_iterations or \'iterations\' in spec)\n        iterations=fixed_iterations or spec.get(\'iterations\',1400)\n        if spec[\'kind\']==\'lgb\':\n            from lightgbm import LGBMRegressor,early_stopping\n            model=LGBMRegressor(n_estimators=iterations,learning_rate=.025,num_leaves=spec.get(\'leaves\',15),min_child_samples=spec.get(\'leaf_samples\',60),reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,subsample_freq=1,random_state=seed,n_jobs=4,verbosity=-1,objective=spec.get(\'objective\',\'regression\'))\n            params=dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],eval_metric=\'rmse\',callbacks=[early_stopping(100,verbose=False,first_metric_only=True)]) if early else {}\n            model.fit(ztr,ys/1e6,**params)\n            trees=model.best_iteration_ if early else iterations\n        else:\n            from xgboost import XGBRegressor\n            params=dict(n_estimators=iterations,max_depth=spec.get(\'depth\',3),learning_rate=.025,min_child_weight=30,reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,random_state=seed,n_jobs=4,tree_method=\'hist\',objective=spec.get(\'objective\',\'reg:squarederror\'),eval_metric=\'rmse\',base_score=float(ys.mean()/1e6))\n            if early: params[\'early_stopping_rounds\']=100\n            if spec.get(\'objective\')==\'reg:pseudohubererror\': params[\'huber_slope\']=12\n            model=XGBRegressor(**params)\n            model.fit(ztr,ys/1e6,**(dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],verbose=False) if early else {}))\n            trees=model.best_iteration+1 if early else iterations\n        p=model.predict(zva)*1e6\n    else:\n        model=sklearn_model(X,spec,seed)\n        model.fit(X.iloc[train_idx],ys/1e6)\n        p=model.predict(X.iloc[valid_idx])*1e6; trees=0\n    if return_fitted:\n        fitted=dict(model=model,preprocessor=pre if spec[\'kind\'] in [\'lgb\',\'xgb\'] else None,scale=1 if spec[\'kind\']==\'cat\' else 1e6)\n        return np.maximum(p,0),trees,fitted\n    return np.maximum(p,0),trees\n\ndef predict_fitted(fitted,X):\n    transformed=fitted[\'preprocessor\'].transform(X) if fitted[\'preprocessor\'] is not None else X\n    return np.maximum(fitted[\'model\'].predict(transformed)*fitted[\'scale\'],0)\n\ndef cat_fit(X, y, spec, seed):\n    params=dict(iterations=spec[\'iterations\'], learning_rate=.04, depth=spec[\'depth\'],\n                l2_leaf_reg=spec.get(\'l2\',5), random_seed=seed, thread_count=2,\n                verbose=False, allow_writing_files=False,\n                loss_function=f"Huber:delta={spec[\'delta\']*1e6:g}" if \'delta\' in spec else \'RMSE\')\n    if spec.get(\'monotone\'):\n        params[\'monotone_constraints\']={**{c:1 for c in SOCIAL+[\'Hygiene Rating\',\'Restaurant Zomato Rating\']},\'Resturant Tier\':-1}\n    if spec.get(\'ordered\'): params[\'boosting_type\']=\'Ordered\'\n    if \'subsample\' in spec: params.update(bootstrap_type=\'Bernoulli\',subsample=spec[\'subsample\'])\n    if \'random_strength\' in spec: params[\'random_strength\']=spec[\'random_strength\']\n    fitted=CatBoostRegressor(**params)\n    target=np.minimum(y,spec[\'cap\']*1e6) if \'cap\' in spec else y\n    fitted.fit(X,target,cat_features=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist())\n    return fitted\n\ndef get_splits(train,scheme):\n    if str(scheme)==\'city\':\n        group=features(train,\'engineered\')[\'City\']\n        splits=list(GroupKFold(5).split(train,groups=group))\n        for tr,va in splits:assert set(group.iloc[tr]).isdisjoint(group.iloc[va])\n    else:splits=list(KFold(5,shuffle=True,random_state=int(scheme)).split(train))\n    counts=np.zeros(len(train),int)\n    for tr,va in splits:\n        assert set(tr).isdisjoint(va);counts[va]+=1\n    assert (counts==1).all()\n    return splits\n\ndef compute_v3(train,y,scheme,contract):\n    splits=get_splits(train,scheme);pred=np.zeros(len(y))\n    for name,spec in contract[\'v2_configs\'].items():\n        if name not in contract[\'v2_weights\']:continue\n        X=features(train,spec[\'variant\']);component=np.zeros(len(y))\n        for f,(tr,va) in enumerate(splits):\n            component[va],_=fit_predict(X,y,tr,va,spec,43+f,spec.get(\'iterations\'))\n        pred+=.5*contract[\'v2_weights\'][name]*component\n    X=features(train,\'base\');spec=contract[\'v3_parameters\']\n    for f,(tr,va) in enumerate(splits):\n        parts=[]\n        for seed in contract[\'v3_seeds\']:\n            model=cat_fit(X.iloc[tr],y[tr],spec,seed+f)\n            parts.append(np.maximum(model.predict(X.iloc[va]),0))\n        pred[va]+=.5*np.mean(parts,axis=0)\n    return pred\n\ndef check_submission(frame,test):\n    assert frame.shape==(500,2),\'Expected 500 rows and two columns\'\n    assert frame.columns.tolist()==[\'Registration Number\',\'Annual Turnover\'],\'Incorrect header\'\n    assert frame[\'Registration Number\'].equals(test[\'Registration Number\']),\'Restaurant IDs/order changed\'\n    assert frame[\'Registration Number\'].is_unique,\'Duplicate restaurant IDs\'\n    assert np.isfinite(frame[\'Annual Turnover\']).all(),\'Non-finite prediction\'\n    assert (frame[\'Annual Turnover\']>=0).all(),\'Negative turnover\'\n\ndef v5_features(raw,variant=\'base\'):\n    X=features(raw,\'engineered\' if variant==\'rich\' else \'base\')\n    if variant in [\'clean\',\'rich\']:\n        X[\'City\']=X[\'City\'].replace({\'bengaluru\':\'bangalore\',\'banglore\':\'bangalore\',\'gurugram\':\'gurgaon\',\'gurgoan\':\'gurgaon\',\'bombay\':\'mumbai\',\'calcutta\':\'kolkata\',\'delhi\':\'new delhi\',\'bhubneshwar\':\'bhubaneswar\',\'hderabad\':\'hyderabad\',\'punr\':\'pune\'})\n    if variant==\'rich\':\n        fb=X[\'Facebook Popularity Quotient\'];ig=X[\'Instagram Popularity Quotient\']\n        X[\'Social ratio\']=fb/(ig+1)\n        X[\'Social product\']=fb*ig\n        X[\'Social minimum\']=np.minimum(fb,ig)\n        X[\'Social absolute gap\']=np.abs(fb-ig)\n        X[\'Quality spread\']=X[[\'Food Rating\',\'Hygiene Rating\',\'Overall Restaurant Rating\']].std(axis=1)\n        X[\'Zomato x Hygiene\']=X[\'Restaurant Zomato Rating\']*X[\'Hygiene Rating\']\n        X[\'City tier x Restaurant tier\']=X[\'Restaurant City Tier\']*X[\'Resturant Tier\']\n    return X\n\ndef inverse_target(z,power):\n    if power==0:return np.expm1(np.minimum(z,20))\n    return np.maximum(z,0)**(1/power)\n\ndef fit_v5(raw,y,spec,seed):\n    X=v5_features(raw,spec.get(\'variant\',\'base\'));target=np.asarray(y,dtype=float)/1e6\n    if \'cap\' in spec:target=np.minimum(target,spec[\'cap\'])\n    power=spec.get(\'power\',1.)\n    transformed=np.log1p(target) if power==0 else target**power\n    loss=spec.get(\'loss\',\'RMSE\')\n    params=dict(iterations=spec.get(\'iterations\',850),learning_rate=.04,depth=spec.get(\'depth\',5),\n                l2_leaf_reg=10,bootstrap_type=\'Bernoulli\',subsample=.7,loss_function=loss,\n                random_seed=seed,thread_count=2,verbose=False,allow_writing_files=False)\n    if \'rsm\' in spec:params[\'rsm\']=spec[\'rsm\']\n    if \'topk\' in spec:\n        pilot=CatBoostRegressor(**{**params,\'iterations\':300})\n        pilot.fit(X,transformed,cat_features=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist())\n        ranked=np.argsort(-pilot.feature_importances_);selected=[X.columns[i] for i in ranked[:spec[\'topk\']]]\n        X=X[selected]\n    model=CatBoostRegressor(**params)\n    model.fit(X,transformed,cat_features=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist())\n    correction=0.\n    if spec.get(\'correct_mean\'):\n        # Training-only retransformation correction; all validation labels remain unseen.\n        fitted_original=inverse_target(model.predict(X),power)\n        correction=float(np.mean(target-fitted_original))\n    return {\'model\':model,\'columns\':X.columns.tolist(),\'power\':power,\'correction\':correction,\'spec\':spec}\n\ndef predict_v5(fitted,raw):\n    X=v5_features(raw,fitted[\'spec\'].get(\'variant\',\'base\'))[fitted[\'columns\']]\n    return np.maximum(0,(inverse_target(fitted[\'model\'].predict(X),fitted[\'power\'])+fitted[\'correction\'])*1e6)\n\ndef evaluate(train,y,scheme,contract,previous=None):\n    if previous is None:previous=compute_v3(train,y,scheme,contract)\n    prediction=contract[\'v3_weight\']*previous\n    parts={}\n    for name,c in contract[\'components\'].items():\n        p=np.zeros(len(y))\n        for f,(tr,va) in enumerate(get_splits(train,scheme)):\n            one=[]\n            for seed in contract[\'seeds\']:\n                fitted=fit_v5(train.iloc[tr],y[tr],c[\'spec\'],seed+f)\n                one.append(predict_v5(fitted,train.iloc[va]))\n            p[va]=np.mean(one,axis=0)\n        parts[name]=p;prediction+=c[\'weight\']*p\n    return {\'previous\':previous,\'candidate\':prediction,**parts}\n\ndef full_fit(train,test,y,contract):\n    prior=np.zeros(len(test));fitted={\'v2\':{},\'v3\':[],\'new\':{}}\n    tr=np.arange(len(train));va=np.arange(len(train),len(train)+len(test));yy=np.r_[y,np.full(len(test),np.nan)]\n    for name,w in contract[\'v2_weights\'].items():\n        spec=contract[\'v2_configs\'][name]\n        X=pd.concat([features(train,spec[\'variant\']),features(test,spec[\'variant\'])],ignore_index=True)\n        p,_,model=fit_predict(X,yy,tr,va,spec,42,spec.get(\'iterations\'),return_fitted=True)\n        fitted[\'v2\'][name]=model;prior+=.5*w*p\n    for seed in contract[\'v3_seeds\']:\n        model=cat_fit(features(train,\'base\'),y,contract[\'v3_parameters\'],seed);fitted[\'v3\'].append(model)\n        prior+=.5*np.maximum(model.predict(features(test,\'base\')),0)/len(contract[\'v3_seeds\'])\n    candidate=contract[\'v3_weight\']*prior\n    for name,c in contract[\'components\'].items():\n        fitted[\'new\'][name]=[]\n        for seed in contract[\'seeds\']:\n            model=fit_v5(train,y,c[\'spec\'],seed);fitted[\'new\'][name].append(model)\n            candidate+=c[\'weight\']*predict_v5(model,test)/len(contract[\'seeds\'])\n    return prior,candidate,fitted\n\ndef predict_final(raw,fitted,contract):\n    prior=sum(.5*w*predict_fitted(fitted[\'v2\'][name],features(raw,contract[\'v2_configs\'][name][\'variant\'])) for name,w in contract[\'v2_weights\'].items())\n    prior+=.5*np.mean([np.maximum(m.predict(features(raw,\'base\')),0) for m in fitted[\'v3\']],axis=0)\n    p=contract[\'v3_weight\']*prior\n    for name,c in contract[\'components\'].items():p+=c[\'weight\']*np.mean([predict_v5(m,raw) for m in fitted[\'new\'][name]],axis=0)\n    return p\n'
ref = types.ModuleType('reference')
exec(compile(REFERENCE_SOURCE, '<frozen v5>', 'exec'), ref.__dict__)
sys.modules['reference'] = ref

In [ ]:
"""V7: turnover bands, partially pooled tier specialists and nested residual models."""
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from scipy.special import softmax
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import reference as ref

REQUIRED = ['Train_dataset.csv', 'Test_dataset.csv', 'Data_Dictionary.csv', 'Sample_Solution.csv']
SEEDS = [10242, 20242]
BANDS = [20., 40., 80.]
SPECS = {}
for depth in [3, 5]:
    for calibrate in [False, True]:
        SPECS[f'bands_d{depth}_cal{int(calibrate)}'] = {
            'family': 'bands', 'depth': depth, 'calibrate': calibrate}
for depth in [4, 6]:
    for shrink in [100, 300]:
        SPECS[f'tiers_d{depth}_shrink{shrink}'] = {
            'family': 'tiers', 'depth': depth, 'shrink': shrink}
for alpha in [100, 1000]:
    SPECS[f'residual_ridge{alpha}'] = {'family': 'residual', 'kind': 'ridge', 'alpha': alpha}
for loss in ['RMSE', 'Huber:delta=20']:
    SPECS['residual_tree_' + ('l2' if loss == 'RMSE' else 'h20')] = {
        'family': 'residual', 'kind': 'cat', 'loss': loss}


def load_data(directory):
    directory = Path(directory)
    assert all((directory / p).is_file() for p in REQUIRED), f'Four official inputs required in {directory}'
    train = pd.read_csv(directory / REQUIRED[0]).rename(columns={'Endorsed By': 'Endoresed By'})
    test = pd.read_csv(directory / REQUIRED[1])
    sample = pd.read_csv(directory / REQUIRED[3])
    pd.read_csv(directory / REQUIRED[2])
    assert len(test) == 500 and train['Registration Number'].is_unique and test['Registration Number'].is_unique
    assert train.drop(columns='Annual Turnover').columns.tolist() == test.columns.tolist()
    assert set(train['Registration Number']).isdisjoint(test['Registration Number'])
    assert sample.columns.tolist() == ['Registration Number', 'Annual Turnover']
    assert sample['Registration Number'].equals(test['Registration Number'])
    y = train['Annual Turnover'].to_numpy(float)
    assert np.isfinite(y).all() and (y > 0).all()
    return train, test, y


def features(raw):
    return ref.v5_features(raw, 'clean')


def split_indices(raw, scheme):
    scheme = str(scheme)
    if scheme == 'city':
        groups = features(raw)['City']
        result = list(GroupKFold(5).split(raw, groups=groups))
        for a, b in result:
            assert set(groups.iloc[a]).isdisjoint(groups.iloc[b])
    elif scheme.startswith('strat:'):
        seed = int(scheme.split(':')[1])
        bins = np.digitize(raw['Annual Turnover'].to_numpy(float) / 1e6, BANDS, right=True)
        result = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(raw, bins))
    elif scheme.startswith('inner:'):
        seed = int(scheme.split(':')[1])
        result = list(KFold(3, shuffle=True, random_state=seed).split(raw))
    else:
        result = list(KFold(5, shuffle=True, random_state=int(scheme)).split(raw))
    counts = np.zeros(len(raw), int)
    for a, b in result:
        assert not set(a).intersection(b)
        counts[b] += 1
    assert (counts == 1).all()
    return result


def cat_params(seed, depth, trees, l2=20):
    return dict(iterations=trees, depth=depth, learning_rate=.04, l2_leaf_reg=l2,
                bootstrap_type='Bernoulli', subsample=.7, random_seed=seed,
                thread_count=2, verbose=False, allow_writing_files=False)


def residual_features(raw, baseline):
    X = features(raw)
    keep = ['Facebook Popularity Quotient', 'Instagram Popularity Quotient',
            'Resturant Tier', 'Hygiene Rating', 'Restaurant Zomato Rating',
            'Restaurant City Tier', 'Opening Year', 'Overall Restaurant Rating',
            'Cuisine', 'Restaurant Type', 'Restaurant Location']
    Z = X[keep].copy()
    Z['Base prediction'] = np.asarray(baseline) / 1e6
    Z['Social mean'] = Z[['Facebook Popularity Quotient', 'Instagram Popularity Quotient']].mean(axis=1)
    Z['Social hygiene'] = Z['Social mean'] * Z['Hygiene Rating'] / 100
    Z['Tier1 base'] = Z['Base prediction'] * (Z['Resturant Tier'] == 1)
    Z['Hygiene base'] = Z['Base prediction'] * (Z['Hygiene Rating'] - 5) / 5
    return Z


def fit_component(raw, y, spec, seed, inner_baseline=None):
    X = features(raw)
    target = np.asarray(y, float) / 1e6
    cats = X.select_dtypes(include=['object', 'string']).columns.tolist()
    info = {'seed': seed, 'train_rows': len(raw), 'family': spec['family']}
    if spec['family'] == 'bands':
        labels = np.digitize(target, BANDS, right=True)
        counts = np.bincount(labels, minlength=4)
        assert (counts >= 3).all()
        centers = np.array([target[labels == k].mean() for k in range(4)])
        temperature = 1.
        if spec['calibrate']:
            prob = np.zeros((len(X), 4))
            for f, (a, b) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed + 411).split(X, labels)):
                model = CatBoostClassifier(**cat_params(seed + 100 + f, spec['depth'], 800), loss_function='MultiClass')
                model.fit(X.iloc[a], labels[a], cat_features=cats)
                assert np.array_equal(model.classes_, np.arange(4))
                prob[b] = model.predict_proba(X.iloc[b])
            log_prob = np.log(np.clip(prob, 1e-12, 1))
            temperatures = [.7, 1., 1.4, 2.]
            losses = [-np.mean(np.log(np.clip(softmax(log_prob / t, axis=1)[np.arange(len(X)), labels], 1e-12, 1))) for t in temperatures]
            temperature = temperatures[int(np.argmin(losses))]
            info.update(temperature_candidates=temperatures, inner_oof_logloss=losses)
        model = CatBoostClassifier(**cat_params(seed, spec['depth'], 800), loss_function='MultiClass')
        model.fit(X, labels, cat_features=cats)
        assert np.array_equal(model.classes_, np.arange(4))
        info.update(band_counts=counts.tolist(), band_centers_millions=centers.tolist(), temperature=temperature)
        return {'family': 'bands', 'model': model, 'centers': centers, 'temperature': temperature, 'info': info}
    if spec['family'] == 'tiers':
        global_model = CatBoostRegressor(**cat_params(seed, 5, 850, 10), loss_function='Huber:delta=40')
        global_model.fit(X, target, cat_features=cats)
        local = {}
        for tier in [1., 2.]:
            mask = X['Resturant Tier'].to_numpy() == tier
            n = int(mask.sum())
            if n < 120:
                continue
            model = CatBoostRegressor(**cat_params(seed + int(tier) * 71, spec['depth'], 650, 30), loss_function='Huber:delta=40')
            model.fit(X.loc[mask], target[mask], cat_features=cats)
            local[tier] = {'model': model, 'weight': n / (n + spec['shrink']), 'rows': n}
        info['local_tiers'] = {str(t): {'weight': value['weight'], 'rows': value['rows']} for t, value in local.items()}
        return {'family': 'tiers', 'global': global_model, 'local': local, 'info': info}
    assert spec['family'] == 'residual' and inner_baseline is not None
    inner_baseline = np.asarray(inner_baseline, float)
    assert inner_baseline.shape == target.shape and np.isfinite(inner_baseline).all()
    Z = residual_features(raw, inner_baseline)
    residual = target - inner_baseline / 1e6
    if spec['kind'] == 'ridge':
        categorical = Z.select_dtypes(include=['object', 'string']).columns.tolist()
        numeric = [c for c in Z if c not in categorical]
        pre = ColumnTransformer([
            ('num', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), numeric),
            ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=10, sparse_output=False), categorical)])
        model = make_pipeline(pre, Ridge(alpha=spec['alpha']))
        model.fit(Z, residual)
    else:
        model = CatBoostRegressor(**cat_params(seed, 2, 300, 50), loss_function=spec['loss'])
        model.fit(Z, residual, cat_features=Z.select_dtypes(include=['object', 'string']).columns.tolist())
    info.update(inner_baseline='three-fold v5 OOF within this outer training fold', mean_inner_residual_millions=float(residual.mean()))
    return {'family': 'residual', 'model': model, 'info': info}


def predict_component(fitted, raw, baseline=None):
    if fitted['family'] == 'bands':
        p = fitted['model'].predict_proba(features(raw))
        p = softmax(np.log(np.clip(p, 1e-12, 1)) / fitted['temperature'], axis=1)
        prediction = p @ fitted['centers'] * 1e6
    elif fitted['family'] == 'tiers':
        X = features(raw)
        prediction = fitted['global'].predict(X)
        for tier, local in fitted['local'].items():
            mask = X['Resturant Tier'].to_numpy() == tier
            if mask.any():
                prediction[mask] = (1 - local['weight']) * prediction[mask] + local['weight'] * local['model'].predict(X.loc[mask])
        prediction *= 1e6
    else:
        assert baseline is not None and len(baseline) == len(raw)
        prediction = np.asarray(baseline) + fitted['model'].predict(residual_features(raw, baseline)) * 1e6
    prediction = np.maximum(prediction, 0)
    assert prediction.shape == (len(raw),) and np.isfinite(prediction).all()
    return prediction


def evaluate_component(raw, y, spec, scheme, baseline, inner_provider=None, progress=None):
    individual = np.zeros((len(y), len(SEEDS)))
    diagnostics = []
    for fold, (a, b) in enumerate(split_indices(raw, scheme)):
        inner = inner_provider(a, fold) if spec['family'] == 'residual' else None
        for j, seed in enumerate(SEEDS):
            fitted = fit_component(raw.iloc[a], y[a], spec, seed + fold, inner)
            individual[b, j] = predict_component(fitted, raw.iloc[b], baseline[b])
            diagnostics.append({'fold': fold, **fitted['info']})
        if progress:
            progress(f'fold {fold + 1}/5 complete')
    return individual.mean(axis=1), individual, diagnostics


def v5_fold_prediction(train, y, valid, contract, fold):
    """Exact legacy outer-fold seed schedule; no validation labels enter fitting."""
    n = len(train)
    a = np.arange(n)
    b = np.arange(n, n + len(valid))
    joined_y = np.r_[np.asarray(y), np.full(len(valid), np.nan)]
    prior = np.zeros(len(valid))
    for name, weight in contract['v2_weights'].items():
        spec = contract['v2_configs'][name]
        X = pd.concat([ref.features(train, spec['variant']), ref.features(valid, spec['variant'])], ignore_index=True)
        p, _ = ref.fit_predict(X, joined_y, a, b, spec, 43 + fold, spec.get('iterations'))
        prior += .5 * weight * p
    for seed in contract['v3_seeds']:
        model = ref.cat_fit(ref.features(train, 'base'), np.asarray(y), contract['v3_parameters'], seed + fold)
        prior += .5 * np.maximum(model.predict(ref.features(valid, 'base')), 0) / len(contract['v3_seeds'])
    p = contract['v3_weight'] * prior
    for name, component in contract['components'].items():
        for seed in contract['seeds']:
            model = ref.fit_v5(train, np.asarray(y), component['spec'], seed + fold)
            p += component['weight'] * ref.predict_v5(model, valid) / len(contract['seeds'])
    return p


def inner_v5_oof(raw, y, contract, seed=731, progress=None):
    prediction = np.zeros(len(raw))
    for f, (a, b) in enumerate(split_indices(raw, f'inner:{seed}')):
        prediction[b] = v5_fold_prediction(raw.iloc[a], np.asarray(y)[a], raw.iloc[b], contract, f)
        if progress:
            progress(f'inner fold {f + 1}/3 complete')
    return prediction


def fit_submission(train, test, y, contract, choice, inner_baseline=None):
    # Fresh baseline, including independent verification against the scored v5 file outside this function.
    _, baseline, legacy = ref.full_fit(train, test, y, contract)
    fitted = {'choice': choice, 'legacy': legacy, 'contract': contract, 'models': []}
    prediction = (1 - choice['weight']) * baseline
    for seed in SEEDS:
        model = fit_component(train, y, SPECS[choice['name']], seed, inner_baseline)
        fitted['models'].append(model)
        prediction += choice['weight'] * predict_component(model, test, baseline) / len(SEEDS)
    return prediction, baseline, fitted


def predict_submission(fitted, raw):
    baseline = ref.predict_final(raw, fitted['legacy'], fitted['contract'])
    choice = fitted['choice']
    component = np.mean([predict_component(m, raw, baseline) for m in fitted['models']], axis=0)
    return (1 - choice['weight']) * baseline + choice['weight'] * component

In [ ]:
CONTRACT = json.loads('{"v2_configs": {"engineered": {"kind": "cat", "variant": "engineered", "depth": 6, "iterations": 260}, "engineered_drop": {"kind": "cat", "variant": "engineered_drop", "depth": 6, "iterations": 216}, "ordered_depth5": {"kind": "cat", "variant": "base", "depth": 5, "boosting_type": "Ordered", "iterations": 219}, "huber20_engineered": {"kind": "cat", "variant": "engineered_drop", "depth": 5, "loss": "Huber:delta=20000000", "iterations": 699}, "huber40": {"kind": "cat", "variant": "base", "depth": 5, "loss": "Huber:delta=40000000", "iterations": 664}, "clip120": {"kind": "cat", "variant": "base", "depth": 5, "cap": 120000000.0, "iterations": 591}, "spline_ridge": {"kind": "spline", "variant": "base", "alpha": 100}, "svr_core": {"kind": "svr", "variant": "core", "C": 20}, "huber40_engineered": {"kind": "cat", "variant": "engineered", "depth": 5, "loss": "Huber:delta=40000000", "iterations": 619}, "huber40_monotone": {"kind": "cat", "variant": "base", "depth": 5, "loss": "Huber:delta=40000000", "monotone": true, "iterations": 447}}, "v2_weights": {"engineered": 0.06448907249408362, "engineered_drop": 0.09631834163153197, "ordered_depth5": 0.08507039177778712, "huber20_engineered": 0.06274452861282048, "huber40": 0.08855362607985372, "clip120": 0.07794631558814363, "spline_ridge": 0.15648185431084338, "svr_core": 0.058428294374292714, "huber40_engineered": 0.09327747966602483, "huber40_monotone": 0.2166900954646185}, "v3_parameters": {"variant": "base", "depth": 5, "iterations": 850, "delta": 40, "subsample": 0.7, "l2": 10}, "v3_seeds": [1042, 2042, 3042], "version": "v5", "v3_weight": 0.25, "seeds": [10242, 20242], "components": {"sqrt_d6_correct1": {"spec": {"power": 0.5, "depth": 6, "correct_mean": true}, "weight": 0.25}, "subspace_h40": {"spec": {"loss": "Huber:delta=40", "rsm": 0.5, "iterations": 1200}, "weight": 0.25}, "quantile0.6": {"spec": {"loss": "Quantile:alpha=0.6", "depth": 5}, "weight": 0.25}}, "development": {"settings": 22, "shortlisted_settings": 6, "splits": [42, 71], "selection": {"name": "top3", "new_weight": 0.75, "rmse42": 19817236.327581998, "rmse71": 19840250.361659143, "mean_rmse": 19828743.34462057, "gain42": 21791.213259298354, "gain71": 29833.246005315334}, "screening_is_optimistic": true}, "confirmation": {"new_splits": [18421, 18422], "previous_splits": [2026, 303], "city_stress": true, "all_splits_reuse_previously_seen_labels": true}}')
CHOICE = json.loads('{"name": "residual_tree_l2", "family": "residual", "weight": 0.75, "rmse_42": 19815813.77766609, "gain_42": 1422.5499159097672, "rmse_71": 19790779.339553073, "gain_71": 49471.02210607007, "mean_rmse": 19803296.558609582, "metrics": {"18421": {"baseline": 19863662.63228201, "candidate": 19832420.144617647, "gain": 31242.48766436428, "relative_change": -0.0015728462692268241}, "18422": {"baseline": 19816733.502104715, "candidate": 19798032.98382704, "gain": 18700.518277674913, "relative_change": -0.0009436730970666662}, "strat:27421": {"baseline": 19828989.57780773, "candidate": 19788419.321618978, "gain": 40570.25618875027, "relative_change": -0.0020460072375122484}, "city": {"baseline": 20136850.755091805, "candidate": 20113476.98948276, "gain": 23373.76560904458, "relative_change": -0.0011607458332646425}}, "local_champion": true, "experimental_eligible": true, "mean_confirmation_gain": 24971.502971019596}')

train, test, y = load_data(DATA_DIR)
changed = train.copy()
changed['Annual Turnover'] = 1e30
changed['Registration Number'] = -7
pd.testing.assert_frame_equal(features(changed), features(train))
assert features(train).columns.equals(features(test).columns)
assert np.isclose(ref.rmse([0, 3], [0, 0]), np.sqrt(4.5))
print('Train:', train.shape, 'Test:', test.shape)
print('PASS: source alignment, metric, target/ID exclusion and feature schema.')

## Actual recorded comparisons

These tables contain the completed development and frozen confirmation results.
They are not new scores from this execution. The next cells retrain one complete
five-fold comparison and the full predictor, then verify their predictions.
All metrics retain the original INR targets. Turnover-band classification is an
internal modeling technique; its output is converted into continuous expected turnover.

In [ ]:
development = pd.DataFrame(json.loads('[{"name":"residual_tree_l2","family":"residual","weight":0.75,"rmse_42":19815813.777666088193655,"gain_42":1422.549915909767151,"rmse_71":19790779.339553073048592,"gain_71":49471.022106070071459,"mean_rmse":19803296.558609582483768},{"name":"residual_tree_l2","family":"residual","weight":0.5,"rmse_42":19810143.860737662762404,"gain_42":7092.466844335198402,"rmse_71":19801518.13877509534359,"gain_71":38732.222884047776461,"mean_rmse":19805830.999756380915642},{"name":"residual_tree_l2","family":"residual","weight":1.0,"rmse_42":19827622.67085687071085,"gain_42":-10386.343274876475334,"rmse_71":19785806.240391347557306,"gain_71":54444.121267795562744,"mean_rmse":19806714.455624110996723},{"name":"residual_tree_l2","family":"residual","weight":0.25,"rmse_42":19810618.191107016056776,"gain_42":6618.13647498190403,"rmse_71":19818013.26526915282011,"gain_71":22237.09638999029994,"mean_rmse":19814315.728188082575798},{"name":"residual_ridge1000","family":"residual","weight":0.5,"rmse_42":19803654.953146729618311,"gain_42":13581.374435272067785,"rmse_71":19834116.038259316235781,"gain_71":6134.32339982688427,"mean_rmse":19818885.495703019201756},{"name":"residual_ridge1000","family":"residual","weight":0.25,"rmse_42":19806903.813850585371256,"gain_42":10332.51373141258955,"rmse_71":19833665.347771022468805,"gain_71":6585.013888120651245,"mean_rmse":19820284.580810803920031},{"name":"residual_ridge100","family":"residual","weight":0.25,"rmse_42":19807359.077969558537006,"gain_42":9877.2496124394238,"rmse_71":19836853.933047097176313,"gain_71":3396.428612045943737,"mean_rmse":19822106.505508329719305},{"name":"residual_ridge1000","family":"residual","weight":0.75,"rmse_42":19807493.231091298162937,"gain_42":9743.096490699797869,"rmse_71":19841601.953688200563192,"gain_71":-1351.592029057443142,"mean_rmse":19824547.59238974750042},{"name":"residual_ridge100","family":"residual","weight":0.5,"rmse_42":19811949.327210303395987,"gain_42":5287.000371694564819,"rmse_71":19847480.547449380159378,"gain_71":-7230.185790237039328,"mean_rmse":19829714.937329843640328},{"name":"tiers_d4_shrink300","family":"tiers","weight":0.25,"rmse_42":19822850.286160863935947,"gain_42":-5613.958578865975142,"rmse_71":19837757.594558399170637,"gain_71":2492.767100743949413,"mean_rmse":19830303.940359629690647},{"name":"tiers_d4_shrink100","family":"tiers","weight":0.25,"rmse_42":19824252.125107683241367,"gain_42":-7015.797525685280562,"rmse_71":19838610.602035608142614,"gain_71":1639.759623534977436,"mean_rmse":19831431.363571643829346},{"name":"tiers_d6_shrink300","family":"tiers","weight":0.25,"rmse_42":19826329.943290822207928,"gain_42":-9093.615708824247122,"rmse_71":19837617.374291192740202,"gain_71":2632.987367950379848,"mean_rmse":19831973.65879100933671},{"name":"residual_tree_h20","family":"residual","weight":0.25,"rmse_42":19820127.007342744618654,"gain_42":-2890.679760746657848,"rmse_71":19844279.740268226712942,"gain_71":-4029.378609083592892,"mean_rmse":19832203.373805489391088},{"name":"tiers_d6_shrink100","family":"tiers","weight":0.25,"rmse_42":19830580.567437816411257,"gain_42":-13344.239855818450451,"rmse_71":19839690.927039716392756,"gain_71":559.434619426727295,"mean_rmse":19835135.747238766402006},{"name":"residual_ridge1000","family":"residual","weight":1.0,"rmse_42":19818414.529937375336885,"gain_42":-1178.20235537737608,"rmse_71":19856115.13704975694418,"gain_71":-15864.775390613824129,"mean_rmse":19837264.833493568003178},{"name":"tiers_d4_shrink300","family":"tiers","weight":0.5,"rmse_42":19836283.291784152388573,"gain_42":-19046.964202154427767,"rmse_71":19843264.426542963832617,"gain_71":-3014.064883820713021,"mean_rmse":19839773.85916355997324},{"name":"residual_tree_h20","family":"residual","weight":0.5,"rmse_42":19828597.939728315919638,"gain_42":-11361.612146317960651,"rmse_71":19853148.17909649759531,"gain_71":-12897.81743735074997,"mean_rmse":19840873.059412404894829},{"name":"tiers_d6_shrink300","family":"tiers","weight":0.5,"rmse_42":19843113.216098330914974,"gain_42":-25876.88851633295053,"rmse_71":19842857.566768903285265,"gain_71":-2607.205109760165215,"mean_rmse":19842985.391433618962765},{"name":"tiers_d4_shrink100","family":"tiers","weight":0.5,"rmse_42":19841077.093715652823448,"gain_42":-23840.766133654862642,"rmse_71":19846941.974788721650839,"gain_71":-6691.613129574805498,"mean_rmse":19844009.534252185374498},{"name":"bands_d5_cal0","family":"bands","weight":0.25,"rmse_42":19841251.155600126832724,"gain_42":-24014.828018128871918,"rmse_71":19850039.621594704687595,"gain_71":-9789.259935561567545,"mean_rmse":19845645.388597417622805},{"name":"bands_d5_cal1","family":"bands","weight":0.25,"rmse_42":19841251.155600126832724,"gain_42":-24014.828018128871918,"rmse_71":19850039.621594704687595,"gain_71":-9789.259935561567545,"mean_rmse":19845645.388597417622805},{"name":"bands_d3_cal0","family":"bands","weight":0.25,"rmse_42":19843863.479427572339773,"gain_42":-26627.151845574378967,"rmse_71":19853046.254976000636816,"gain_71":-12795.893316857516766,"mean_rmse":19848454.867201782763004},{"name":"bands_d3_cal1","family":"bands","weight":0.25,"rmse_42":19843863.479427572339773,"gain_42":-26627.151845574378967,"rmse_71":19853046.254976000636816,"gain_71":-12795.893316857516766,"mean_rmse":19848454.867201782763004},{"name":"tiers_d6_shrink100","family":"tiers","weight":0.5,"rmse_42":19853684.49210699275136,"gain_42":-36448.164524994790554,"rmse_71":19849015.48061503842473,"gain_71":-8765.11895589902997,"mean_rmse":19851349.986361019313335},{"name":"residual_ridge100","family":"residual","weight":0.75,"rmse_42":19830997.029000144451857,"gain_42":-13760.70141814649287,"rmse_71":19872107.708401784300804,"gain_71":-31857.346742641180754,"mean_rmse":19851552.368700966238976},{"name":"residual_tree_h20","family":"residual","weight":0.75,"rmse_42":19842641.978018898516893,"gain_42":-25405.650436900556088,"rmse_71":19866849.197765190154314,"gain_71":-26598.836106050759554,"mean_rmse":19854745.587892048060894},{"name":"tiers_d4_shrink300","family":"tiers","weight":0.75,"rmse_42":19857519.476418439298868,"gain_42":-40283.148836441338062,"rmse_71":19856764.202081214636564,"gain_71":-16513.840422071516514,"mean_rmse":19857141.839249826967716},{"name":"tiers_d6_shrink300","family":"tiers","weight":0.75,"rmse_42":19867566.658331196755171,"gain_42":-50330.330749198794365,"rmse_71":19855964.705654822289944,"gain_71":-15714.343995679169893,"mean_rmse":19861765.681993007659912},{"name":"tiers_d4_shrink100","family":"tiers","weight":0.75,"rmse_42":19867686.312704600393772,"gain_42":-50449.985122602432966,"rmse_71":19865231.934402570128441,"gain_71":-24981.57274342700839,"mean_rmse":19866459.123553585261106},{"name":"residual_tree_h20","family":"residual","weight":1.0,"rmse_42":19862247.300457127392292,"gain_42":-45010.972875129431486,"rmse_71":19885372.807336617261171,"gain_71":-45122.445677474141121,"mean_rmse":19873810.053896870464087},{"name":"tiers_d6_shrink100","family":"tiers","weight":0.75,"rmse_42":19886514.085547961294651,"gain_42":-69277.757965963333845,"rmse_71":19868210.106122203171253,"gain_71":-27959.744463060051203,"mean_rmse":19877362.095835078507662},{"name":"tiers_d4_shrink300","family":"tiers","weight":1.0,"rmse_42":19886533.841793928295374,"gain_42":-69297.514211930334568,"rmse_71":19878240.636598225682974,"gain_71":-37990.274939082562923,"mean_rmse":19882387.239196076989174},{"name":"bands_d5_cal1","family":"bands","weight":0.5,"rmse_42":19888842.260411120951176,"gain_42":-71605.93282912299037,"rmse_71":19882650.738462023437023,"gain_71":-42400.376802880316973,"mean_rmse":19885746.499436572194099},{"name":"bands_d5_cal0","family":"bands","weight":0.5,"rmse_42":19888842.260411120951176,"gain_42":-71605.93282912299037,"rmse_71":19882650.738462023437023,"gain_71":-42400.376802880316973,"mean_rmse":19885746.499436572194099},{"name":"residual_ridge100","family":"residual","weight":1.0,"rmse_42":19864460.594348259270191,"gain_42":-47224.266766261309385,"rmse_71":19910683.464910663664341,"gain_71":-70433.103251520544291,"mean_rmse":19887572.029629461467266},{"name":"tiers_d6_shrink300","family":"tiers","weight":1.0,"rmse_42":19899661.99384406581521,"gain_42":-82425.666262067854404,"rmse_71":19876923.228210557252169,"gain_71":-36672.866551414132118,"mean_rmse":19888292.61102731153369},{"name":"bands_d3_cal1","family":"bands","weight":0.5,"rmse_42":19894601.560009811073542,"gain_42":-77365.232427813112736,"rmse_71":19890509.58468808978796,"gain_71":-50259.223028942942619,"mean_rmse":19892555.572348948568106},{"name":"bands_d3_cal0","family":"bands","weight":0.5,"rmse_42":19894601.560009811073542,"gain_42":-77365.232427813112736,"rmse_71":19890509.58468808978796,"gain_71":-50259.223028942942619,"mean_rmse":19892555.572348948568106},{"name":"tiers_d4_shrink100","family":"tiers","weight":1.0,"rmse_42":19904040.541146971285343,"gain_42":-86804.213564973324537,"rmse_71":19893453.01324362680316,"gain_71":-53202.651584483683109,"mean_rmse":19898746.777195297181606},{"name":"tiers_d6_shrink100","family":"tiers","weight":1.0,"rmse_42":19929021.283779736608267,"gain_42":-111784.956197738632909,"rmse_71":19897246.238975502550602,"gain_71":-56995.877316359430552,"mean_rmse":19913133.761377617716789},{"name":"bands_d5_cal1","family":"bands","weight":0.75,"rmse_42":19959841.000944171100855,"gain_42":-142604.673362173140049,"rmse_71":19937971.728329595178366,"gain_71":-97721.366670452058315,"mean_rmse":19948906.36463688313961},{"name":"bands_d5_cal0","family":"bands","weight":0.75,"rmse_42":19959841.000944171100855,"gain_42":-142604.673362173140049,"rmse_71":19937971.728329595178366,"gain_71":-97721.366670452058315,"mean_rmse":19948906.36463688313961},{"name":"bands_d3_cal1","family":"bands","weight":0.75,"rmse_42":19969266.78642437979579,"gain_42":-152030.458842381834984,"rmse_71":19952501.402640510350466,"gain_71":-112251.040981370955706,"mean_rmse":19960884.094532445073128},{"name":"bands_d3_cal0","family":"bands","weight":0.75,"rmse_42":19969266.78642437979579,"gain_42":-152030.458842381834984,"rmse_71":19952501.402640510350466,"gain_71":-112251.040981370955706,"mean_rmse":19960884.094532445073128},{"name":"bands_d5_cal1","family":"bands","weight":1.0,"rmse_42":20053998.763088952749968,"gain_42":-236762.435506954789162,"rmse_71":20015814.291077364236116,"gain_71":-175563.929418221116066,"mean_rmse":20034906.527083158493042},{"name":"bands_d5_cal0","family":"bands","weight":1.0,"rmse_42":20053998.763088952749968,"gain_42":-236762.435506954789162,"rmse_71":20015814.291077364236116,"gain_71":-175563.929418221116066,"mean_rmse":20034906.527083158493042},{"name":"bands_d3_cal1","family":"bands","weight":1.0,"rmse_42":20067592.084196362644434,"gain_42":-250355.756614364683628,"rmse_71":20038794.066850073635578,"gain_71":-198543.705190930515528,"mean_rmse":20053193.075523220002651},{"name":"bands_d3_cal0","family":"bands","weight":1.0,"rmse_42":20067592.084196362644434,"gain_42":-250355.756614364683628,"rmse_71":20038794.066850073635578,"gain_71":-198543.705190930515528,"mean_rmse":20053193.075523220002651}]'))
confirmation = pd.DataFrame(json.loads('[{"name":"residual_tree_l2","family":"residual","weight":0.75,"scheme":"18421","v5_rmse":19863662.632282011210918,"candidate_rmse":19832420.144617646932602,"gain":31242.487664364278316},{"name":"residual_tree_l2","family":"residual","weight":0.75,"scheme":"18422","v5_rmse":19816733.502104714512825,"candidate_rmse":19798032.983827039599419,"gain":18700.518277674913406},{"name":"residual_tree_l2","family":"residual","weight":0.75,"scheme":"strat:27421","v5_rmse":19828989.577807728201151,"candidate_rmse":19788419.321618974208832,"gain":40570.256188750267029},{"name":"residual_tree_l2","family":"residual","weight":0.75,"scheme":"city","v5_rmse":20136850.755091805011034,"candidate_rmse":20113476.989482760429382,"gain":23373.765609044581652},{"name":"tiers_d4_shrink300","family":"tiers","weight":0.25,"scheme":"18421","v5_rmse":19863662.632282011210918,"candidate_rmse":19868640.34168042242527,"gain":-4977.709398414939642},{"name":"tiers_d4_shrink300","family":"tiers","weight":0.25,"scheme":"18422","v5_rmse":19816733.502104714512825,"candidate_rmse":19819044.127502579241991,"gain":-2310.625397864729166},{"name":"tiers_d4_shrink300","family":"tiers","weight":0.25,"scheme":"strat:27421","v5_rmse":19828989.577807728201151,"candidate_rmse":19834990.790661588311195,"gain":-6001.212853860110044},{"name":"tiers_d4_shrink300","family":"tiers","weight":0.25,"scheme":"city","v5_rmse":20136850.755091805011034,"candidate_rmse":20147551.177671190351248,"gain":-10700.422579389065504},{"name":"bands_d5_cal0","family":"bands","weight":0.25,"scheme":"18421","v5_rmse":19863662.632282011210918,"candidate_rmse":19882829.421373464167118,"gain":-19166.7890914529562},{"name":"bands_d5_cal0","family":"bands","weight":0.25,"scheme":"18422","v5_rmse":19816733.502104714512825,"candidate_rmse":19840755.328738793730736,"gain":-24021.826634079214273},{"name":"bands_d5_cal0","family":"bands","weight":0.25,"scheme":"strat:27421","v5_rmse":19828989.577807728201151,"candidate_rmse":19841828.976922731846571,"gain":-12839.399115003643601},{"name":"bands_d5_cal0","family":"bands","weight":0.25,"scheme":"city","v5_rmse":20136850.755091805011034,"candidate_rmse":20100616.034163746982813,"gain":36234.720928058028221}]'))

best = development.sort_values('mean_rmse').groupby('family', sort=False).head(1)
print('Best development setting per family:')
print(best[['name', 'weight', 'gain_42', 'gain_71']].to_string(index=False))
print('Confirmation for this candidate:')
print(confirmation[confirmation['name'] == CHOICE['name']].to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
positions = np.arange(len(best))
axes[0].barh(positions - .16, best['gain_42'] / 1000, height=.3, label='42', color='#2563eb')
axes[0].barh(positions + .16, best['gain_71'] / 1000, height=.3, label='71', color='#0f766e')
axes[0].set_yticks(positions, best['family'])
axes[0].axvline(0, color='#334155')
axes[0].set(xlabel='RMSE improvement over v5 (INR thousands)', title='Recorded development results')
axes[0].legend(title='Assignment')
axes[1].hist(y / 1e6, bins=45, color='#2563eb', edgecolor='white')
axes[1].set(xlabel='Annual turnover (INR millions)', ylabel='Restaurants', yscale='log', title='All original labels remain in scoring')
fig.tight_layout()
plt.show()

## Fresh five-fold reproduction, with resumable checkpoints

Compressed reference arrays are used only to check freshly trained results. They
are never substituted for model predictions. Existing checkpoints are reused only
when input, source, configuration, fold and file hashes match. Delete the checkpoint
folder or set `REUSE_CHECKPOINTS=False` for a completely fresh repetition.

Residual models generate three-fold v5 predictions inside each outer training fold.
No outer validation labels enter the residual model or its preprocessing.

In [ ]:
EXPECTED_BASELINE_OOF = np.frombuffer(zlib.decompress(base64.b64decode('...')), dtype='<f8').copy()
EXPECTED_COMPONENT_OOF = np.frombuffer(zlib.decompress(base64.b64decode('...')), dtype='<f8').copy()
SOURCE_HASHES = {'reference': '1a24f9f116170ea54bd3242264d5cd2a150c3429e039a50cb11c3112e52247c1', 'models': '029de9731d057d8827ff6c7f1789f7669b408eeb3626e50a1677db223614edf8'}

REUSE_CHECKPOINTS = os.environ.get('V7_REPRODUCTION_FORCE_FRESH') != '1'
checkpoint_root = OUTPUT_DIR / 'validation_checkpoints'
checkpoint_root.mkdir(parents=True, exist_ok=True)
cache_reads = 0
fresh_baseline = np.zeros(len(y))
fresh_component = np.zeros(len(y))
for fold, (a, b) in enumerate(split_indices(train, 18421)):
    checkpoint = checkpoint_root / f'fold_{fold}.npz'
    meta = checkpoint.with_suffix('.json')
    context = {'inputs': INPUT_HASHES, 'sources': SOURCE_HASHES, 'choice': CHOICE,
               'contract': CONTRACT, 'packages': PACKAGES, 'fold': fold,
               'train_ids': train.iloc[a]['Registration Number'].tolist(),
               'valid_ids': train.iloc[b]['Registration Number'].tolist()}
    if REUSE_CHECKPOINTS and checkpoint.exists() and meta.exists():
        saved = json.loads(meta.read_text())
        assert saved['context'] == context
        assert saved['sha256'] == hashlib.sha256(checkpoint.read_bytes()).hexdigest()
        cached = np.load(checkpoint, allow_pickle=False)
        baseline_part, component_part = cached['baseline'], cached['component']
        cache_reads += 1
        print('Verified checkpoint:', fold)
    else:
        print('Fresh outer fold:', fold + 1, flush=True)
        baseline_part = v5_fold_prediction(train.iloc[a], y[a], train.iloc[b], CONTRACT, fold)
        inner = inner_v5_oof(train.iloc[a], y[a], CONTRACT, 731 + fold, print) if CHOICE['family'] == 'residual' else None
        one = []
        for seed in SEEDS:
            model = fit_component(train.iloc[a], y[a], SPECS[CHOICE['name']], seed + fold, inner)
            one.append(predict_component(model, train.iloc[b], baseline_part))
        component_part = np.mean(one, axis=0)
        temporary = checkpoint.with_suffix('.partial.npz')
        np.savez_compressed(temporary, baseline=baseline_part, component=component_part)
        temporary.replace(checkpoint)
        meta.write_text(json.dumps({'context': context, 'sha256': hashlib.sha256(checkpoint.read_bytes()).hexdigest()}))
    fresh_baseline[b], fresh_component[b] = baseline_part, component_part
assert np.max(np.abs(fresh_baseline - EXPECTED_BASELINE_OOF)) < .01
assert np.max(np.abs(fresh_component - EXPECTED_COMPONENT_OOF)) < .01
fresh_selected = (1 - CHOICE['weight']) * fresh_baseline + CHOICE['weight'] * fresh_component
print('PASS: complete five-fold prediction reproduction within INR 0.01; reused checkpoints:', cache_reads)
fresh_segments = []
for name, mask in [('all', np.ones(len(y), bool)), ('<=40M', y <= 40e6),
                   ('40-80M', (y > 40e6) & (y <= 80e6)), ('>80M', y > 80e6)]:
    fresh_segments.append({'segment': name, 'rows': int(mask.sum()),
        'v5_rmse': ref.rmse(y[mask], fresh_baseline[mask]),
        'candidate_rmse': ref.rmse(y[mask], fresh_selected[mask])})
print(pd.DataFrame(fresh_segments).to_string(index=False))

## Fresh final fit and CSV checks

Every selected full-data model is trained again. Expected test predictions are
reproduction references, not hidden test targets. The score bound below assumes
the same 500 rows and ordinary RMSE stated in the competition instructions; it is
a necessary condition for beating the shown leader, not evidence of accuracy.

In [ ]:
def check_behavior(fitted, test, prediction):
    permutation = np.random.default_rng(91).permutation(len(test))
    assert np.max(np.abs(predict_submission(fitted, test.iloc[permutation]) - prediction[permutation])) < .01
    edge = test.iloc[:3].copy()
    edge['City'] = 'unseen_city_v7'
    edge['Cuisine'] = pd.NA
    edge['Facebook Popularity Quotient'] = np.nan
    edge['Resturant Tier'] = np.nan
    assert np.isfinite(predict_submission(fitted, edge)).all()
    changed = test.copy()
    changed['Registration Number'] = -777
    changed['Annual Turnover'] = 1e30
    assert np.max(np.abs(predict_submission(fitted, changed) - prediction)) < .01
    frame = pd.DataFrame({'Registration Number': test['Registration Number'], 'Annual Turnover': prediction})
    ref.check_submission(frame, test)
    invalid = [frame.iloc[:-1], pd.concat([frame, frame.iloc[:1]], ignore_index=True),
               frame.iloc[::-1].reset_index(drop=True), frame.iloc[:, ::-1],
               frame.rename(columns={'Annual Turnover': 'Turnover'}), frame.assign(index=0)]
    for column, value in [('Registration Number', 'submission'), ('Registration Number', test['Registration Number'].iloc[1]),
                           ('Annual Turnover', np.nan), ('Annual Turnover', np.inf), ('Annual Turnover', -np.inf), ('Annual Turnover', -1.)]:
        bad = frame.copy()
        if isinstance(value, str):
            bad[column] = bad[column].astype(object)
        bad.loc[0, column] = value
        invalid.append(bad)
    for bad in invalid:
        try:
            ref.check_submission(bad, test)
        except (AssertionError, ValueError, TypeError):
            continue
        raise AssertionError('Malformed submission accepted')
    return {'rows': len(test), 'columns': frame.columns.tolist(), 'finite_nonnegative': True,
            'row_order': True, 'unique_ids': True, 'permutation_invariance': True,
            'target_id_invariance': True, 'missing_unseen_values': True,
            'malformed_cases_rejected': len(invalid)}

In [ ]:
EXPECTED_TEST = np.frombuffer(zlib.decompress(base64.b64decode('eJwNl3c41f8bxoVUJKJdpKJllPFDkxulQkMpSjSklHGO4+z1OcMhoTIavkiFUDLSJJVImZWyykhFRbKTxO/z//u63s91X8/9uu9H66dfm4IBH8e/OmUvnCLBQPpqseMOLh5E7HeJnxgMxTn5L2p7ZQhKVk8cGhNB/2iAqYE2AeeTthOSLQgMv4w7OlDNhXVv+hOFDSz0NGxQzp3BxWhz/qpdtgT86353vDYRYWfl8nBGLB10zzdjXyoDESantagqSwLvvS6lVcVirH9/KlR2zA/O71U3J3qS/2TkFFwjuJhs3PMy6TkFupS1jmu9+HCtnLjweAsTux8evZ59nY5b0Tfmj2aKIT842DpS6Q850Rztlh1iWDokbJJEiBBj8s0aV/igNH3oUFcJwry5b87sdZbim2biplp9MX5cZu0uucoBUfHzREypCBeWijUOy4LgYX/L518iH9T5O7x/n5BgrfJ6z0d2BPZ0KVPvrwhCKMXh88wpBCIoIxGduRzkhRcP3/rBgfimrWTxbDZ4tP3Xo13ZUNJ/bxz/WYzQGd++3DMPhBd3olMrj41DmYvWVg6LcFszY1BukIEFI9tPFpXzYb9n57mp6hJklrjZeI9LMVmtwsOoj4kjzYo/r9uzMa9jpvKje2KkSUdaal4L4fD4WOn6RhouLGHZe2oKYfTS5rylmxBh/75bNnjz4BF+PCk9PBBDe8qvly8U4MevhJYRoRDJ0mlxOk0ySJV1qlsMBehak74jq45ArnbnCz9FGhJr/kr9z/hBZ+vZMuNWFmY2fvprOFmG9NvTh6VeDLwR/3Ib0RMivkLyR+8QC8n5Ob8OzqYjUtv0GJNgY+o9ndKBIj5YG+/VOukJ4NetYlnXJIRj2H9+Y5l83NJR3zDlKwtfrRWvfjMlwAl27OsMInArfKn3y0YOvD6nic5TRegrndePwzy4b/24UbuNB93p50oZlmykJxd1aFmJ4DjrrIc1qW+i14ZRm8cCFK1Q0/TK4KK4dsX5rsUBiDNQszz0hYmr91frul4JQK1E/K52qhjiTvqxETs+Fu9rt7Iy5MB11lsL45VsXGp4G3P8ohBrDw61RK1hwv2QJj1KwkfPJ78wiZII3Amu6+NrCXyaFhxgTupwTsta7XuNBK87rg8YdUix80Rh/KskDswb+lsiW/jIc3dF0SQBLn1Z7W/uQYOq98QJev0ErhUUmL7aLkaXNGuHvSYdJ2VEWMxMMcrePBNlPhNBKB81weKIDB4OPY+5dCouf3/51UkmQU96beyBbjoKaL2PZP9k8G9cJ2u6T0W7uX5ZxBM+rmmMXth/nIfhuhcWcroUaGSV/311OxS8xqfl1R/YOD5sl1s1xkCn4nlT/U2BeJhKkaWeFmLqY25a4B4x6r+5u8vC6ajfcuvIokgGrN5cmS65yEFrWp/ZsUYCia9qTbJCBHDK8izQmcpCW1245/AcEe796Qx2y+DB5T1qqnREmCvMf1fSzkZCX8jiYZEIQXyRUcgyIbwWrz6pLRHiV+i8yGcSOhQLr1VHv2XBPr7pTjuNh5JRkVeLZzC8jproeufxsDJv38XNhnwMf3J7Y/SHjfbsEO++vyL0/jBVpawKhLNPts2dZTR0t32rXL+f5I6Rs+1Ucp4TSh+3PKRRUPA3+1WamIeicjc3fgnpN6Uj0zLMOZBqH0q86iyE5dlcStk10kd2yhPbKtmYNc9rfMdFMVTGze80xcrQkeq9568zgbgNDfS2+2LskdisHhKTfthTph/YT3Jp4rbqjWfEiPNZuZP6geSf5uM08cdg2J1R8HldTcCkZOKpuUMMXCbmdx34SaCua4uZ8gUu9hTbWHW1EDB9+qXAbbEA3iM137NjmVBsVapQy6Nj78KQBSpvCKifebvZqyoYX/yfFrhYEdha1fzjtYCJ7isZzFfyQnQlfHN8pMTFWM4+WfoPOhZZ9zeOH6YiUutyS4xiIGiDrcUnsglsWJ/7vrhMisjSr5pFylQk/rz1Ob6UgLeoRxr3iY29n+x1nxSIUPxxsa5GOgsBZ8rSaysYUP5jHuVmKET4Rtchj1EalLace2phSKBAU2nHhXgJWoS2XfS7QvjWHi5bOS6E8pK+MWV5Bvbbfki7EcPG5v4q1pz9NOSKtnX37pIgckWIxUozIVrPtXb6qAugGUnNlx4RoG3wg3qSkA77odgUlxl0uNi/Cxpo5qPwYmNVha4Y0l5+spI7DcnFT5ZOmBGEoCXNW1JfUCCdJWfxc4UItdF3OZnnpCg+7GMyIJBie9V0QdppEWq2P9ZcaMDC1QyVBZJ+H3R/MfjcoMCAdm6VT5UHEy+XKHSn1EngGDRff5MPD17nu6QTDoqQ/SR8s00RFx06rao35Qn08DyXP6oVgDJq3dzkQPJnp5lZUo4I4+HUW4vySN8Qse+OrBTjptySh11jAiz56/rUBDzsq4tYz3CXQEfJIqlejYdg+bRG1zo2Ylb4HnM8K4P8P00dxlEqVKR3DmZdZaGsKvnZcw0pMreulUa+o2Dqujb1BC0hXgXsOynrpOPUf0/U0wf9UV/2zn7/dSHsDR1O00PoEHzhHnbsJvCvc+3uT/JiBEUVdX6bTvKhwojh9paBQEfVNOMvQiS+rPyfRxsTDTV2W1ckc7BXYY7fyUk0PEuferohhwD1XqaC3jsZdj7MGhrMDkJaSLTD7LcEmpVjbrvvpUDunutFP0Mp3D0ZXvxDQTB6ZWRQTeVAqPs9SzWPCf29htI/ZL59v9hlmThEh1FcfkyjjwCjOb2r1BYIYV64fdX4agmefqtzPreej3NX6qxKMhjoiwtzGCH1Gh0f+5+hB4HXvYn77pwUw9dC9WHHMypkQZZ6jmcJ2Puubw49SKCQ2t7Ofi9Bym2/COdoJmySNs6cs5CA47ohSlQxA6+NS3J5qWwY2A9GmPT6oatrnt7MdQTaZ9dIbXiB2Lq0aE1iCQu/UzUT72txsE9n1UD/b3J/znf9O6cmwZMMkw3VBiK4rzJ/ToRzEL/uGmMvQeBvmq3tuRw+lLVrNcTbmDBYYO/AtqfAfP57xbMEE6uXK/mH/RPDZGZsydlfXISFnlzXoxCIiuH8J0vJnNuvvOaaqRITJjaGGlN0GKinRX2ojOHBsdyjOOxaEFg6P+zerRLh4uM1tAJVOuQLi/0f/EfmTFXTL8WdNFQkTblx/i4NBlfuLAupJvvKgxspH7ok6FLrjenfxYR/hHaw9Rby/QSz4qzOINgQuleLuAzU6KXnz69nI6580+eamxxEh1XkPtCiwvFg7vUlQyLM0orzM3choDP5yNbSK1xY+q5wf072pJXdanPuk3kRaj7hgoTMh4V+P1eJTFnw5fsFCcspcJpNMZCeJ/mbUJ6iT+7p+rH+ccuZbCxdoxSenyjEvNnfxNN2SuEakja5P1SG37wTKf+ZsGDsWH/n61oubPp/nfF2EiFgXcyObeM09J0qvxmnEYKCuxpqCedFiKptqppOkHswcvnoSg4V2RMiOq5oiSDD6aFAFQ4aDZ/atVtRMSM45OyJ7RLc0Tl9WzVdjKpvm1xCvhNYzXl+y89RCgb9ctUsTwb2Srsn3yRzZZjZWDTgysSstw7LPowQEPoaPFDKYWKv7taGT210rHqtnhd1nPTN+xX0o6YhGFduG9MIkODP5scnW2cRkMt4njBG9r2eGgX5LFNS192b0hMKeRiPrVf1n8PGgZIfj9akiMA75reA/pSKHU87hm8GsqE+I7Z6yVM6el2I1a5ubESurMjnBchQa2bkpj9VhDHeGvnj9SGYrevVV/RDDJZJ+mkiUAAVbQuHQpKPhYKjhvbNXNCGubY1UTT8J73bPHpRgISmBYMKN8UwOq+zXX4FDxT/6nyqHAevVpxXNuXKUD/667nF9SBYj/c2Rk0icPd1p4rlyTD8oyz/apdM5oT9p9MNk3ngGK2a1JbKQF7EP0r0vECwKhe1Gj/hAUJrh3GFCFDUPqtWyNGw+ZHRXxMVGaoDL6xV3cdAxSX2rO9OvkhaYtWq8FUMs5Ac53JDHh5SHDSCJQIkV/AqXaIJsA/uMJuZRe77r8hJB+h0vIpI4egt4sN/+U5NzRoe8sqD7Kc8JHto/gu3pdrkvO+MHUsUxShS0dT4YC+DblZkcgbJoaUeDc4dxXTM8H4U53RTiGjJvLwjvoG4VGuZemAKHTopqWuU6AKc630xpaGHjtTXpXu637Pw47vF5wobBmbYzNlVdoDkj1Pp/3JU2YivnKhvWcbGBfm4F0p8sqdX2qYI+wRQq1WNwXY+irNxbF+9CK5XDDfr0Am0DNftjzYSomrjuP3lgzQUnOk3W0fyq2/CHtZjKxY0n+36bDQggldw1+cILwHud+tcEmlQERxb+en+LhbeNvMrZzqQ/SryWmRKshi6hAq0bWlwuWKz2/W5ABmlm38pJ0lR71Z/yZMtxqHBKfG7J0rxY+Cqk62nGA8G7nEbrIMwbvl9nvdiJr7Y0k6FvuChxtQvw76DSvawRAWWCh8NNtLCAHUmxJ8XsJs8WDixTOVmjjAQz1NE/8440aCh/6Vda3UwMkN5I+1kT9Wrk7vhQ+bpguO7Wox7qdjy5KKKek8QrKhVRu71XPy+ZezmbStDeBct1T/MF9oz30+8sVqGaU+KOvZc4iK8v/3m8p8cLPjgUd0Ty8GUpvmnL5I8dK0BXw9U6NDW+Gwg78iVHkRLVlMg5nzXmRx1VQKPoIO8S28lqG5m6+3+6Y9kM2aW2SkK1DaoaJXzJTh0aenox41cbE7PdBvO5OBpoYXfw9sEUu8vcS4l+7fHocZrIQmBWLSwo/+IEoH0hN1HZiicgOXGpcaSdTzUXVY5PfsSyTk7k5wt26QY6LLS1npB5pHHqU6bGhFM5/46UBlFoIM+UWp8jcDZs5fjN90NwWKTq9lXKFz4mIkn6VP8YNOuGPHRjYq5dUe1BwN4iLbrSLIZYaG/YNn9wnIm2hx3aGdnkXfc/q752oo8zPU3mn+0iI3JdOo/uwAOfFUvHTxA6vx2jmaodJoQ7YNqxx+n8ZCes7iYu56AByfFV48mxcakV48ULDn4P5SPUrg=')), dtype='<f8').copy()
EXPECTED_V5 = np.frombuffer(zlib.decompress(base64.b64decode('eJwNl2k41VsUxguZ0nAzE1e3FFEplXQjL0kiSlLUVYjCwZnn4X/OMZUGhMhQUVRSIXOlkSKlEpWZroqbClEU93zaH/azn732Xu9612819xxpvrlWgMTtrl/6+yQgVwZMfnLlYVH01cQ1RCTOHUlRGJ4bDQOW9ZHL7WLscVtnIOgQwaGtyuzCXhGGBCUdN2bzMaGtmpe7iw2j8TdaClo8ZHabXHayJbDNOnWJjaLsnFDBdn49AxFNXstS/6NCp460KT1eCk2ytd3RxxL8EZD2sWQkFOcCC0wNN4uxXq8pRDWKh/yCnJihs2T0nFY0XXdYALdrja0Gd1j41U1SOVjGwGiHbd3THAm+XnRfvkgzHG0H5W1p1hKk5yZ8y48VI/epeXsOUwDqcre8g450XCVttPHVjITvzpy6cGMJTuR5/gi7xoWhYoya5lkxjh/o0T9WREdLLzujwVwADcdHP9S8pch2H+zIW0NggKqe57GYDmHuk/DXf4sQVPvxUUEfFz8T1VcQs3nIGXkZuN+dgwDFzLVVehwEHehfzauSYFPW6/xacyq2TnsYEsDmYK3Id7T1vRgWfj26oXNZeJYwfDy4VIBbxZEBcQpSxNpvIBm0RuLL6fYLFYksfC+S66bTODjJvHompFyCDV9nc1Jk//8g6fAQbQYdRbWWzU8fCnF4wwIdepAIT1vXEPVxfOQXXvNKvkGBuW905ZSxEJ9nJV65clSEtsCCtHu50Ugquv/zoo4Q4yazSoOeEEgcXekHXRo8VP6JsWsNBX8PXbt8gg2dAEXuwaoo8G6tWnFXyoST5RydhN8CBCwou2B6lA0rk/G+LF0GpK8fJmrwOViql3WqX6Yr/+ZceedZQjzcQFr2Z6MIC78Mh3mfEGCv8i63tB1sTM2dkfFiMYF27wjOyB4Ctx5F+lc94aJJm3im6iuG1ZUbXwbpfIS3cNekN/KxLEFHedtGDhaNns8KsRDj8cu65Lt8CXLfOex7fFv2fqnfzGUZPPhlD/XNlobDpjY/a/03FtJnSX/rSSLQrXi3/L6CLO+deT9tDAWw67NmV/pyYT0Ss2yhtyzu2gp4JIgwdXzvVlIQC/tva8+vkghwpmCF8Y9eApdJcYP1rQQGT8fwRgVMXH+f1Rh2UIrKHZdYPV6RcD703uBHKRf+nffEqi8EuKH9Q6F3iRA1jjsHTvjRsNmwed/1AQISncw5FBsJxCo5po9UGVgzapw1X1WCrtX7/G7VitG8+UGXgmY07iaws5WpFCwNi4s8xpJiOPWXFyWPge27zNeUxEfj6vUPaiNxFCxvSBsRVArgFBtQdf4kH0dCUx9UaJBxOv1D2tmGGBh9rDBIVuCC0+UUSl3OQnmkzcNeOypO6rikc9JEkOPaynVvlUBJS/Ii5TwDr8IvtzEYTAz/47FqRR4XsTMarmc+ImD9tl4pmSuE57W0ihQ1FnpvPsfATwJPOn6Ysuv5+O+ZZOFfSmJU1jod7FXhIrvTo5SQiGGUNrfwXqYA49+qHXJk+uvOXvQ4MIQOaV7/ruRJNk5znvZ8YfMR+tpf/ntbJERv7EkcCR9O0fM/2BgL0Bbka/DbhAsebf+Nyi9iLBxepLrflopprwamvdlAQ6XcgvuC4zzM14/46qMqRvial8n918gIoG8rK73IhyBwc9LdYjFU8rXsdv6W5XnKwu2JjQiTVjc9tPJEkNjyP36YxsXZV1FvEhIkKF96wYPOj8arE680Lv1DoMpR7d/ZhyUQuse5Rvmz8blnwrFGjwILjx3DrzkSGJb6dl+U4+DYmTMpg/1RsEur1jUzIRC9ZUhn9xgTdYy1ZeRSAqnji+/26fCQtmDFliNvCSTxn8wLHBBgrs3LEHILC9xhS/LOJgaWU6u/tsv2V1Nme2eqRoFe5h+pbEUgpX9hsns6C3mPCk8sVBfhneruKJ9xLlYnNTzv1mPC5E5SCMWbgsPpDEGABRUP+uYcLWYRKPdseU3KjYTF1J7IU7soyOjZZbxAFp/N7G/k6AkODKQ7zVc7i0He5zZW3s9Gl/nvOnd1Jh6pprr9OiCEuVeb+s9fNPgEfxiKSBYhRT4lo89Min5VruKdGhES3SJnmKsR2J2tH1ChxkSNT117DsFBbFZZzSwDGpod9DZMfZbgl0bdx1/yMn+y4/18L/OFgpddSYtpQpQseXm88QYDS8cLspvmMDCRNPrJtUGAliLf6osqEpAyTPzUTGmYfP8pn98pWzsspZ5eZPjXVNj0LBBjVujm75p+kYgfmd52mhoJXp+uwtMtYoRsj5k8a8dG9POQU/ddgtExHKvsVMqAkDaoNSVloYTarGH5lxTVFRYtT5342Gg2fv2OvRjlhN+zb4k83N7aqD4xnUDdoetuyzuE8FPP/GhzjINsIf/L7wQxwnrP2rZnSLD7Oe38qJIEYw6ZOzzniBBaKRUtMOJjWHVhnbqrFMTPF+qXdPgolztxSF/MQZ299ylPv2gcSv5103ExBZ8MWYWOT9hYEpr6r+ncSJitm/Chy/retq/dicXTRBD9kzAW+JyBfzcXdr5JDsfjiOk6aREiVCXoLpuoYaB96tmjJJle7rrFrz/wUoxzT9buLFOKwkhwdX9xDhNdAy9VvxeIkORmkkBfyQbZfDjm+X0u3tgsL/gym4ZH01epTiURCCnZlb+2IBr7a4+p3Sqkg984q+BsDYG4gemUB5pk/Pn5VHaSUiT+iFfq/vaDhusUPf8MBhcxOmcJd5l+/5Dy2sjVHKy7IHhpvZ4Jz78z7rlAiPIg+8/as0W4sCXNZ+CTBA0b9N13rhIg9YHXX8cHmNCqP7q0U8YNW5UUby7xIMCtC16jESGBlmF0uNZPCr47PzGgGhJQDK9+Q1iJYEWq4Ny4LMWtYUcJ6yoDWQt636dMIxDlxLuk9p2JOTQB27qfA1eNiHe3V4ci9K/N26atJbD93DR7/m0q5nu0HHfoYkPX/n36TTIHk5kRX5pPRqL36L29uVMSBD+zrDHSEiOK4uB8qJAL6rJOqxYSgWtuGrdLiwX4Y/CHu4uQhRbl36EhM8nIlcshtWWyYGJcc85uuhTSyTH9bAU+Tl6ydbZ+TcHY+YfRr2T1++L7p5kNBixMPB+Mf2HGxIQSzdIxko+HPU+VyCV0vOqvUltuLEbBxv7TPsoMvFbeHTJ2hwnfZcXnlkpoeKX//Xv/Yxq+WX0ddmoQw+5YCkv/mRRvkw126jozEXwkvoJziIl5cYZ5F7poSN1AnvCmMXHzGt+w5y0Hhy3/7LI7xoXxvJUcs78pUNbx6Z01JYa/XoFLZyAB/bXnhWF3eDDbsvBjSrkUq3JytA3NWKjsvUQlVkpwam+ZUrCAjXL91evifpHx6malbkcADQc1ejdvUBHhQNtjqzyZn9+svVNceVGEF0fYVEfLSNDnhI8d4USjSXPxYnsSGyeytItGN/IwNJJzLumATK95yp7KTnTYF9/xmbcnGovyV1hVu4nRYTSt8GuqBFtffbgd70ZB21af7zMMxfBxkS9SceXAqEe866qirH4OvWvs2yhFvcpeZVexBPf0rxve7yHgeH6o/Y+rMs6sWmz1S8REwq6qnkulfJjXHlIP82LhLGfHf+3fZD4d/mbd0s8sNJHqNj7oYuDEUYv/PgcxYV0YWfzMMAYzxhYdUN4vhXzgvv7uGQR0smfMSdrJhUAC/kUHDux03j1xJ/HxdOJx++R+DjYPNDrUJ8v6YJ2frXMmBTqN1X87hXNwPjwgwqaWAb1kFlEs4zk5uRRXgXc0WmerWLqNyervzkyXTX/HgBHbMiOmTgL9q//gjax+ikcM7nmRGDB1Njic+54HrWkXFthV0UC5xNHYuFSI8xNG6zoLZT60b812/XEeuq26lMhzuTjs/XXai8BoBDM/63hX0aHNWJstTRKh+t/WdIfjR+CbrGH5tYoKwb6wlT9X8eBKeZ2a/oCJqJnaORs0qXhn+aEo6zEfg0lnHHgJx2T91PYasYmGecafmWFTUTiSMHetxyYmxMlvlydPhmCRs/OLJTcleNaf3n/fmQ/3rWrznNoFcA0V5vvMI3DQf0mByzU+ll6X6L8+yoA56bhNgZEAc4bCxl748REy4qHlW0bgUG6wt6knHQeNj02q/RTjUsR8YdyOKGzfa+Da5kpGs7TZ491DBn552vtYFovw6KzzQEowFfrjvvfTXBgIW3FjsT9JCNUf9d1iLSb6xa1zJ7vZaIwebRoIYKJyiJlxkCSbC05xb58z4SD88gPT0loOGFNTW/r2UKH30zc+o1OIt785Xl6OMm6pWVW35K0YXsEjVrfMCMhxJs0v2wtBvhWb0uhIQ1uFZ3mRBx9EYFx0mBsboRnHYn2axOBovni33EQIql6VotEHMq5zsvwHyWx0m+p+3LeIiedeZp/eMSWoDslYOeQv4+7jWo7b4oWY3hzTnZoSiYPrCxMUgyT4sVjrMG1MCtcfE2or7CWYpGknNwfRoXYF3XvsWAjprXXx3MnH73B5XeZsKrZvjfEfdRAg5Yt03cKNLFh/1Bs6v5mFGY1v/aPSqLAWGm83kfHomdMB2m5/RqHDrWtZsTsfd/1D9WxVZPxp8VpdQYmKV5sDpvKXM+CqbaF4rZuH1dK90xt0o2G+Z72rNJSEK+fiTffKyTh0r0PLPgkPG0/bmHKW8KA50bJQ9wwXtz6s7LLolHFufWMR5RAFA3PuBop7xTA5nfswdqbsfs0oy0fqUqwb79zjvUKKo9et5bTrwvEEJWaeIWRUxOaLSRpSPAgbOvSOwcOKPxkmqVlcCKuUAkczCMwXGT04dVcKM8GaJXHnqKiLHdy3Tp5AjZybx22VIAx+Musf2sJH77iCHdtSxn/9OxbNn5SiNOuNd6JsHjFQrViRmSfG7peLbL+dJFCpVdZQn0igWxBB/70nBsrMvqFdYTy08m1Ui9JDYSi/R7DNlQJyU0m9NIMPesmVKsZ6Dkr8Ll5qbGUhvul44eNdAvQWHUtI0uLDr/CT++kaDp7NWLnp37NcyM/7ajv3PwE2TNfVd5ZxQpmxS2a2gA+SWbmaGgiIm7IsEmQc81ZpXbGHCxf/A9SCQ5w=')), dtype='<f8').copy()

full_inner = inner_v5_oof(train, y, CONTRACT, 731, print) if CHOICE['family'] == 'residual' else None
test_prediction, baseline_test, fitted = fit_submission(train, test, y, CONTRACT, CHOICE, full_inner)
maximum_full_fit_difference = float(np.max(np.abs(test_prediction - EXPECTED_TEST)))
assert maximum_full_fit_difference < .01
assert np.max(np.abs(baseline_test - EXPECTED_V5)) < .01
checks = check_behavior(fitted, test, test_prediction)
output_path = OUTPUT_DIR / 'submission.csv'
pd.DataFrame({'Registration Number': test['Registration Number'], 'Annual Turnover': test_prediction}).to_csv(output_path, index=False)
roundtrip = pd.read_csv(output_path)
ref.check_submission(roundtrip, test)
assert np.max(np.abs(roundtrip['Annual Turnover'].to_numpy() - test_prediction)) < .01
assert len(output_path.read_text().splitlines()) == 501
rms_change = float(np.sqrt(np.mean((test_prediction - baseline_test) ** 2)))
lower_bound = max(0, 12046722.3691139 - rms_change)
print('PASS: fresh final fit, CSV round-trip and behavioral checks.')
print('Maximum full-fit difference (INR):', maximum_full_fit_difference)
print('Checks:', json.dumps(checks, indent=2))
print('CSV SHA256:', hashlib.sha256(output_path.read_bytes()).hexdigest())
print('Prediction RMS change from v5:', rms_change)
print('Conditional portal RMSE lower bound:', lower_bound)
print('Can mathematically beat the shown 4.7967M leader:', lower_bound < 4796740.5600053)
print('Actual candidate portal RMSE: UNKNOWN until submitted.')
print(roundtrip.head().to_string(index=False))

## Choosing the final submission

Upload this trial's CSV to the hackathon, then record its returned RMSE and file
hash. Retain the lowest observed portal score, including the preserved v5 result.
Submit only the matching final executed notebook to the one-time coding assignment
after that choice. Local cross-validation scores do not determine a portal rank.
All validation assignments reuse labels examined in previous work.

In [ ]:
try:
    from google.colab import files
except ImportError:
    print('Local file:', output_path)
else:
    files.download(str(output_path))